In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import r2_score
from scipy.stats import pearsonr, spearmanr
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')


In [2]:
_cost = {
    "localdiffdock": 407.5, "diffdock": 407.5, "flexx": 3.33, "smina": 99.9,
    "gnina": 105.8, "plants": 6.85, "cnnscore": 0.31, "cnnaffinity": 0.31,
    "smina_affinity": 0.31, "ad4": 0.28, "linf9": 0.24, "rtmscore": 0.41,
    "vinardo": 0.29, "scorch": 4.63, "hyde": 2.0, "chemplp": 0.121,
    "rfscore_v1": 0.682, "rfscore_v2": 0.687, "rfscore_v3": 0.69,
    "vina_hydrophobic": 0.69, "vina_intra_hydrophobic": 0.69,
}

print("Loading and Aggregating Data...")
df_orig = pd.read_csv("all_rescoring_results_merged.csv")

agg_rules = {
    'CNNscore': 'max', 'CNNaffinity': 'max', 'smina_affinity': 'max',
    'RTMScore': 'max', 'SCORCH': 'max', 'HYDE': 'max', 'rfscore_v2': 'max',
    'CHEMPLP': 'min', 'vina_hydrophobic': 'min', 'vina_intra_hydrophobic': 'min',
    'true_value': 'first', 'activity_class': 'first'
}

for col in df_orig.select_dtypes(include=np.number).columns:
    if col not in agg_rules and col not in ["pose", "id", "true_value", "activity_class"]:
        agg_rules[col] = "mean"

df_agg = df_orig.groupby(["id","docking_tool"]).agg(agg_rules).reset_index()

meta_cols = ["true_value", "activity_class"]
score_cols = [c for c in df_agg.columns if c not in meta_cols and c not in ["id","docking_tool"]]
wide = df_agg.set_index(["id", "docking_tool"])[score_cols].unstack("docking_tool")
wide.columns = [f"{tool}_{score}" for score, tool in wide.columns]
meta = df_agg.groupby("id")[meta_cols].first()
df_matrix = wide.join(meta)

# nan_rows = df_matrix[df_matrix.isna().any(axis=1)]
# nan_rows

Loading and Aggregating Data...


In [4]:
import pandas as pd
import numpy as np
import itertools
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

# 1. Define Costs Dictionary
costs = {
    "localdiffdock": 407.5, "diffdock": 407.5, "flexx": 3.33, "smina": 99.9,
    "gnina": 105.8, "plants": 6.85, "cnnscore": 0.31, "cnnaffinity": 0.31,
    "smina_affinity": 0.31, "ad4": 0.28, "linf9": 0.24, "rtmscore": 0.41,
    "vinardo": 0.29, "scorch": 4.63, "hyde": 2.0, "chemplp": 0.121,
    "rfscore_v1": 0.682, "rfscore_v2": 0.687, "rfscore_v3": 0.69,
    "vina_hydrophobic": 0.69, "vina_intra_hydrophobic": 0.69,
}
# 2. Setup the Data and Target (Assuming df_matrix is already loaded)
X_full = df_matrix.drop(columns=["true_value", "activity_class"])
y = df_matrix["true_value"]

base_tools = ['diffdock', 'flexx', 'gnina', 'plants', 'smina']

# 3. Define the XGBoost Model & CV
xgb_model = XGBRegressor(
    max_depth=4, learning_rate=0.05, n_estimators=300,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def calculate_subset_cost(columns):
    total_cost = 0
    used_tools = set()
    for col in columns:
        parts = col.split('_', 1)
        if len(parts) == 2:
            tool, score = parts[0].lower(), parts[1].lower()
            used_tools.add(tool)
            total_cost += costs.get(score, 0)
    for tool in used_tools:
        total_cost += costs.get(tool, 0)
    return total_cost

# 4. Generate Combinations and Evaluate
results = []
print("Starting Multi-Metric Combination Analysis...")

for r in range(1, len(base_tools) + 1):
    for combo in itertools.combinations(base_tools, r):
        combo_name = " + ".join(combo)
        subset_cols = [col for col in X_full.columns if any(col.startswith(tool + '_') for tool in combo)]
        
        if not subset_cols:
            continue
            
        X_subset = X_full[subset_cols]
        combo_cost = calculate_subset_cost(subset_cols)
        
        # Lists to store metrics for each of the 5 folds
        fold_rmse, fold_mae, fold_r2, fold_pearson, fold_spearman = [], [], [], [], []
        
        # Explicit Cross-Validation Loop
        for train_idx, test_idx in cv.split(X_subset):
            X_train, X_test = X_subset.iloc[train_idx], X_subset.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            # Train and Predict
            xgb_model.fit(X_train, y_train)
            y_pred = xgb_model.predict(X_test)
            
            # Calculate all metrics
            fold_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
            fold_mae.append(mean_absolute_error(y_test, y_pred))
            fold_r2.append(r2_score(y_test, y_pred))
            
            # Scipy returns a tuple: (correlation_coefficient, p_value) - we only need [0]
            fold_pearson.append(pearsonr(y_test, y_pred)[0])
            fold_spearman.append(spearmanr(y_test, y_pred)[0])
            
        # Store the mean of the 5 folds
        results.append({
            "Tool Combination": combo_name,
            "Total Cost per Molecule": round(combo_cost, 2),
            "RMSE": np.mean(fold_rmse),
            "RMSE-HIGH": np.max(fold_rmse),
            "RMSE-LOW": np.min(fold_rmse),
            "MAE": np.mean(fold_mae),
            "R2": np.mean(fold_r2),
            "R2-HIGH": np.max(fold_r2),
            "R2-LOW": np.min(fold_r2),
            "Pearson (r)": np.mean(fold_pearson),
            "Spearman (rho)": np.mean(fold_spearman)
        })
        
        print(f"Evaluated: {combo_name} | Cost: {round(combo_cost, 2)} | Spearman: {np.mean(fold_spearman):.4f}")

# 5. Format and Output
results_df = pd.DataFrame(results)

# Sort by Spearman (Highest Rank Correlation first)
results_df = results_df.sort_values(by="Spearman (rho)", ascending=False).reset_index(drop=True)

print("\n--- Final Analysis Complete ---")
display(results_df)

Starting Multi-Metric Combination Analysis...
Evaluated: diffdock | Cost: 417.66 | Spearman: 0.3794
Evaluated: flexx | Cost: 13.49 | Spearman: 0.0683
Evaluated: gnina | Cost: 115.96 | Spearman: 0.2360
Evaluated: plants | Cost: 17.01 | Spearman: 0.1851
Evaluated: smina | Cost: 110.06 | Spearman: 0.3412
Evaluated: diffdock + flexx | Cost: 431.15 | Spearman: 0.3496
Evaluated: diffdock + gnina | Cost: 533.62 | Spearman: 0.3780
Evaluated: diffdock + plants | Cost: 434.67 | Spearman: 0.3621
Evaluated: diffdock + smina | Cost: 527.72 | Spearman: 0.4295
Evaluated: flexx + gnina | Cost: 129.45 | Spearman: 0.1895
Evaluated: flexx + plants | Cost: 30.5 | Spearman: 0.1443
Evaluated: flexx + smina | Cost: 123.55 | Spearman: 0.2371
Evaluated: gnina + plants | Cost: 132.97 | Spearman: 0.2478
Evaluated: gnina + smina | Cost: 226.02 | Spearman: 0.2989
Evaluated: plants + smina | Cost: 127.07 | Spearman: 0.3097
Evaluated: diffdock + flexx + gnina | Cost: 547.1 | Spearman: 0.3189
Evaluated: diffdock + fl

,Tool Combination,Total Cost per Molecule,RMSE,RMSE-HIGH,RMSE-LOW,MAE,R2,R2-HIGH,R2-LOW,Pearson (r),Spearman (rho)
0,diffdock + smina,527.72,1.316280,1.496654,1.165903,1.065516,0.154842,0.408480,-0.082458,0.479577,0.429541
1,diffdock + gnina + smina,643.67,1.308321,1.458088,1.165078,1.067327,0.159939,0.409317,-0.103427,0.489705,0.423330
2,diffdock + plants + smina,544.72,1.316725,1.480908,1.190938,1.072908,0.150299,0.382805,-0.099305,0.476981,0.381623
3,diffdock + gnina + plants,550.62,1.328536,1.465224,1.230723,1.061117,0.136061,0.340879,-0.067568,0.459560,0.380797
4,diffdock,417.66,1.334460,1.418209,1.153160,1.082772,0.125415,0.306042,-0.231417,0.456724,0.379411
5,diffdock + gnina + plants + smina,660.68,1.328640,1.452586,1.211341,1.081710,0.129468,0.361476,-0.193434,0.468783,0.378145
6,diffdock + gnina,533.62,1.314135,1.443800,1.200123,1.069811,0.148497,0.373248,-0.108642,0.475021,0.377963
7,diffdock + flexx + gnina + smina,657.16,1.354880,1.450734,1.206624,1.081219,0.084987,0.366439,-0.323786,0.443962,0.374330
8,diffdock + flexx + gnina + plants + smina,674.17,1.335202,1.445893,1.152209,1.067608,0.111822,0.422293,-0.250591,0.464796,0.371709
9,diffdock + plants,434.67,1.322911,1.422169,1.239149,1.060698,0.138561,0.323352,-0.114291,0.467468,0.362065


In [ ]:
import pandas as pd
import numpy as np
import itertools
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_selection import SelectFromModel

# 1. Define Costs Dictionary
costs = {
    "localdiffdock": 407.5, "diffdock": 407.5, "flexx": 3.33, "smina": 99.9,
    "gnina": 105.8, "plants": 6.85, "cnnscore": 0.31, "cnnaffinity": 0.31,
    "smina_affinity": 0.31, "ad4": 0.28, "linf9": 0.24, "rtmscore": 0.41,
    "vinardo": 0.29, "scorch": 4.63, "hyde": 2.0, "chemplp": 0.121,
    "rfscore_v1": 0.682, "rfscore_v2": 0.687, "rfscore_v3": 0.69,
    "vina_hydrophobic": 0.69, "vina_intra_hydrophobic": 0.69,
}

# 2. Setup Data
X_full = df_matrix.drop(columns=["true_value", "activity_class"])
y = df_matrix["true_value"]

base_tools = ['diffdock', 'flexx', 'gnina', 'plants', 'smina']

# 3. Model & CV Setup
xgb_model = XGBRegressor(
    max_depth=4, learning_rate=0.05, n_estimators=300,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Smart Cost Function for Raw Columns Only
def calculate_smart_cost(selected_cols):
    total_cost = 0
    used_tools = set()
    
    for col in selected_cols:
        parts = col.split('_', 1)
        if len(parts) == 2:
            tool, score = parts[0].lower(), parts[1].lower()
            used_tools.add(tool)
            total_cost += costs.get(score, 0)
            
    for tool in used_tools:
        total_cost += costs.get(tool, 0)
        
    return total_cost

# 4. Generate Combinations, Filter Noise, and Evaluate
results = []
print("Starting Clean Baseline Pipeline (Raw Features Only)...")

for r in range(1, len(base_tools) + 1):
    for combo in itertools.combinations(base_tools, r):
        combo_name = " + ".join(combo)
        
        # Select strictly the raw columns for these tools
        subset_cols = [col for col in X_full.columns if any(col.startswith(tool + '_') for tool in combo)]
        if not subset_cols: continue
        X_combo = X_full[subset_cols].copy()
                    
        # ML Feature Selection: Drop raw columns that are just noise
        selector_model = XGBRegressor(max_depth=3, n_estimators=100, random_state=42, n_jobs=-1)
        selector_model.fit(X_combo, y)
        
        selector = SelectFromModel(selector_model, prefit=True, threshold='mean')
        selected_mask = selector.get_support()
        selected_features = X_combo.columns[selected_mask].tolist()
        
        if len(selected_features) == 0: selected_features = X_combo.columns.tolist()
            
        X_selected = X_combo[selected_features]
        combo_cost = calculate_smart_cost(selected_features)
        
        fold_rmse, fold_mae, fold_r2, fold_pearson, fold_spearman = [], [], [], [], []
        
        for train_idx, test_idx in cv.split(X_selected):
            X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            xgb_model.fit(X_train, y_train)
            y_pred = xgb_model.predict(X_test)
            
            fold_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
            fold_mae.append(mean_absolute_error(y_test, y_pred))
            fold_r2.append(r2_score(y_test, y_pred))
            fold_pearson.append(pearsonr(y_test, y_pred)[0])
            fold_spearman.append(spearmanr(y_test, y_pred)[0])
            
        results.append({
            "Tool Combination": combo_name,
            "Cost per Molecule": round(combo_cost, 2),
            "Features Kept": len(selected_features),
            "RMSE": np.mean(fold_rmse),
            "R2": np.mean(fold_r2),
            "Spearman": np.mean(fold_spearman), 
            "Selected Columns List": selected_features 
        })
        
        print(f"Evaluated: {combo_name} | Cost: {round(combo_cost, 2)} | RMSE: {np.mean(fold_rmse):.4f}")

# 5. Output
results_df = pd.DataFrame(results)

# Sort strictly by the lowest RMSE (Best accuracy for exact affinity prediction)
results_df = results_df.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("\n--- Clean Baseline Analysis Complete ---")
pd.set_option('display.max_colwidth', None)
display(results_df)

Starting Clean Baseline Pipeline (Raw Features Only)...
Evaluated: diffdock | Cost: 408.03 | RMSE: 1.5092
Evaluated: flexx | Cost: 5.81 | RMSE: 1.5868
Evaluated: gnina | Cost: 112.34 | RMSE: 1.4533
Evaluated: plants | Cost: 12.89 | RMSE: 1.4358
Evaluated: smina | Cost: 105.63 | RMSE: 1.3604
Evaluated: diffdock + flexx | Cost: 412.62 | RMSE: 1.4504
Evaluated: diffdock + gnina | Cost: 520.44 | RMSE: 1.2544
Evaluated: diffdock + plants | Cost: 417.57 | RMSE: 1.2995
Evaluated: diffdock + smina | Cost: 513.44 | RMSE: 1.3044
Evaluated: flexx + gnina | Cost: 121.33 | RMSE: 1.4594
Evaluated: flexx + plants | Cost: 15.06 | RMSE: 1.5471
Evaluated: flexx + smina | Cost: 115.28 | RMSE: 1.4384
Evaluated: gnina + plants | Cost: 124.73 | RMSE: 1.4173
Evaluated: gnina + smina | Cost: 211.74 | RMSE: 1.4063
Evaluated: plants + smina | Cost: 114.48 | RMSE: 1.4156
Evaluated: diffdock + flexx + gnina | Cost: 520.52 | RMSE: 1.3556
Evaluated: diffdock + flexx + plants | Cost: 431.42 | RMSE: 1.3041
Evaluated:

,Tool Combination,Cost per Molecule,Features Kept,RMSE,R2,Spearman,Selected Columns List
0,diffdock + gnina,520.44,6,1.254446,0.235709,0.462310,"[gnina_smina_affinity, diffdock_RTMScore, gnina_RTMScore, gnina_SCORCH, gnina_vina_hydrophobic, diffdock_vina_intra_hydrophobic]"
1,diffdock + plants,417.57,4,1.299542,0.177507,0.445271,"[diffdock_RTMScore, diffdock_HYDE, plants_CHEMPLP, plants_vina_intra_hydrophobic]"
2,diffdock + flexx + plants,431.42,13,1.304125,0.167621,0.413335,"[diffdock_CNNscore, diffdock_RTMScore, flexx_RTMScore, plants_RTMScore, diffdock_SCORCH, diffdock_HYDE, plants_HYDE, flexx_rfscore_v2, plants_rfscore_v2, flexx_CHEMPLP, diffdock_vina_intra_hydrophobic, flexx_vina_intra_hydrophobic, plants_vina_intra_hydrophobic]"
3,diffdock + smina,513.44,4,1.304434,0.152258,0.446650,"[smina_CNNaffinity, diffdock_RTMScore, smina_SCORCH, smina_rfscore_v2]"
4,diffdock + flexx + gnina + plants + smina,644.93,14,1.309769,0.164111,0.394441,"[gnina_CNNaffinity, diffdock_RTMScore, gnina_SCORCH, plants_SCORCH, smina_SCORCH, plants_HYDE, diffdock_rfscore_v2, gnina_rfscore_v2, flexx_CHEMPLP, gnina_vina_hydrophobic, smina_vina_hydrophobic, diffdock_vina_intra_hydrophobic, flexx_vina_intra_hydrophobic, plants_vina_intra_hydrophobic]"
5,diffdock + gnina + plants + smina,640.41,12,1.313326,0.147226,0.445544,"[diffdock_CNNaffinity, diffdock_smina_affinity, diffdock_RTMScore, gnina_SCORCH, plants_SCORCH, smina_SCORCH, diffdock_HYDE, gnina_rfscore_v2, plants_rfscore_v2, gnina_vina_hydrophobic, smina_vina_hydrophobic, plants_vina_intra_hydrophobic]"
6,diffdock + flexx + gnina + plants,539.85,13,1.313551,0.156146,0.384102,"[gnina_CNNaffinity, diffdock_RTMScore, flexx_RTMScore, plants_RTMScore, gnina_SCORCH, plants_SCORCH, plants_HYDE, gnina_rfscore_v2, plants_rfscore_v2, flexx_CHEMPLP, gnina_vina_hydrophobic, flexx_vina_intra_hydrophobic, plants_vina_intra_hydrophobic]"
7,diffdock + gnina + smina,622.31,9,1.328948,0.141660,0.399967,"[diffdock_CNNaffinity, diffdock_smina_affinity, diffdock_RTMScore, smina_SCORCH, gnina_rfscore_v2, gnina_vina_hydrophobic, smina_vina_hydrophobic, diffdock_vina_intra_hydrophobic, smina_vina_intra_hydrophobic]"
8,diffdock + plants + smina,523.29,7,1.330477,0.129599,0.403353,"[smina_CNNaffinity, diffdock_smina_affinity, diffdock_RTMScore, smina_SCORCH, diffdock_HYDE, smina_vina_hydrophobic, plants_vina_intra_hydrophobic]"
9,diffdock + flexx + smina,518.77,5,1.341059,0.126375,0.411283,"[smina_CNNaffinity, diffdock_RTMScore, smina_SCORCH, diffdock_HYDE, flexx_vina_intra_hydrophobic]"


In [3]:
import pandas as pd
import numpy as np
import itertools
from scipy.stats import spearmanr

# 1. Identify all base tools
base_tools = ['diffdock', 'flexx', 'gnina', 'plants', 'smina']

# 2. Automatically extract all the underlying scoring function names
all_cols = [c for c in df_matrix.columns if c not in ["true_value", "activity_class"]]
score_types = set()
for col in all_cols:
    parts = col.split('_', 1)
    if len(parts) == 2:
        score_types.add(parts[1])

# 3. Calculate Apples-to-Apples Alignments
alignment_results = []

for tool_A, tool_B in itertools.combinations(base_tools, 2):
    pair_corrs = []
    shared_details = []
    
    # Check every possible scoring function
    for stype in score_types:
        col_A = f"{tool_A}_{stype}"
        col_B = f"{tool_B}_{stype}"
        
        # If BOTH tools calculated this exact score...
        if col_A in df_matrix.columns and col_B in df_matrix.columns:
            # Drop missing values (NaNs) just for this pair so the math works
            valid_data = df_matrix[[col_A, col_B]].dropna()
            
            if len(valid_data) > 10: # Ensure we have enough data points to be statistically valid
                # Calculate Spearman correlation for this exact score
                corr = spearmanr(valid_data[col_A], valid_data[col_B])[0]
                pair_corrs.append(corr)
                
                # Format a string to show the user exactly how this score aligned
                shared_details.append(f"{stype}: {corr:.2f}")
                
    # If they shared any scores, calculate the general alignment
    if pair_corrs:
        overall_alignment = np.mean(pair_corrs)
        alignment_results.append({
            "Tool Pair": f"{tool_A} vs {tool_B}",
            "Overall Alignment Fraction": round(overall_alignment, 3),
            "Scores Compared": len(pair_corrs),
            "Individual Score Alignments": " | ".join(shared_details)
        })

# 4. Format and Output
alignment_df = pd.DataFrame(alignment_results)

# Sort by highest Overall Alignment (Most Redundant pairs at the top)
alignment_df = alignment_df.sort_values(by="Overall Alignment Fraction", ascending=False).reset_index(drop=True)

print("--- Apples-to-Apples Tool Alignment ---")
display(alignment_df)

--- Apples-to-Apples Tool Alignment ---


,Tool Pair,Overall Alignment Fraction,Scores Compared,Individual Score Alignments
0,gnina vs smina,0.948,10,smina_affinity: 0.91 | vina_hydrophobic: 0.92 ...
1,gnina vs plants,0.623,10,smina_affinity: -0.09 | vina_hydrophobic: 0.70...
2,plants vs smina,0.617,10,smina_affinity: -0.12 | vina_hydrophobic: 0.70...
3,flexx vs plants,0.582,10,smina_affinity: 0.42 | vina_hydrophobic: 0.50 ...
4,flexx vs smina,0.555,10,smina_affinity: -0.02 | vina_hydrophobic: 0.62...
5,flexx vs gnina,0.550,10,smina_affinity: -0.01 | vina_hydrophobic: 0.61...
6,diffdock vs plants,0.489,10,smina_affinity: 0.35 | vina_hydrophobic: 0.29 ...
7,diffdock vs smina,0.478,10,smina_affinity: -0.23 | vina_hydrophobic: 0.35...
8,diffdock vs gnina,0.463,10,smina_affinity: -0.24 | vina_hydrophobic: 0.32...
9,diffdock vs flexx,0.438,10,smina_affinity: 0.04 | vina_hydrophobic: 0.13 ...


In [5]:
# Tell Pandas to never cut off text in columns
pd.set_option('display.max_colwidth', None)

display(alignment_df)

,Tool Pair,Overall Alignment Fraction,Scores Compared,Individual Score Alignments
0,gnina vs smina,0.948,10,smina_affinity: 0.91 | vina_hydrophobic: 0.92 | HYDE: 0.83 | rfscore_v2: 0.99 | CNNaffinity: 1.00 | vina_intra_hydrophobic: 0.99 | SCORCH: 0.96 | RTMScore: 0.92 | CHEMPLP: 0.97 | CNNscore: 1.00
1,gnina vs plants,0.623,10,smina_affinity: -0.09 | vina_hydrophobic: 0.70 | HYDE: 0.23 | rfscore_v2: 0.89 | CNNaffinity: 0.89 | vina_intra_hydrophobic: 0.89 | SCORCH: 0.62 | RTMScore: 0.72 | CHEMPLP: 0.74 | CNNscore: 0.63
2,plants vs smina,0.617,10,smina_affinity: -0.12 | vina_hydrophobic: 0.70 | HYDE: 0.24 | rfscore_v2: 0.90 | CNNaffinity: 0.90 | vina_intra_hydrophobic: 0.90 | SCORCH: 0.63 | RTMScore: 0.69 | CHEMPLP: 0.73 | CNNscore: 0.62
3,flexx vs plants,0.582,10,smina_affinity: 0.42 | vina_hydrophobic: 0.50 | HYDE: 0.38 | rfscore_v2: 0.80 | CNNaffinity: 0.64 | vina_intra_hydrophobic: 0.87 | SCORCH: 0.60 | RTMScore: 0.43 | CHEMPLP: 0.53 | CNNscore: 0.66
4,flexx vs smina,0.555,10,smina_affinity: -0.02 | vina_hydrophobic: 0.62 | HYDE: 0.31 | rfscore_v2: 0.86 | CNNaffinity: 0.67 | vina_intra_hydrophobic: 0.85 | SCORCH: 0.55 | RTMScore: 0.48 | CHEMPLP: 0.75 | CNNscore: 0.49
5,flexx vs gnina,0.550,10,smina_affinity: -0.01 | vina_hydrophobic: 0.61 | HYDE: 0.30 | rfscore_v2: 0.86 | CNNaffinity: 0.67 | vina_intra_hydrophobic: 0.84 | SCORCH: 0.53 | RTMScore: 0.47 | CHEMPLP: 0.74 | CNNscore: 0.49
6,diffdock vs plants,0.489,10,smina_affinity: 0.35 | vina_hydrophobic: 0.29 | HYDE: 0.17 | rfscore_v2: 0.91 | CNNaffinity: 0.75 | vina_intra_hydrophobic: 0.80 | SCORCH: 0.27 | RTMScore: 0.64 | CHEMPLP: 0.54 | CNNscore: 0.17
7,diffdock vs smina,0.478,10,smina_affinity: -0.23 | vina_hydrophobic: 0.35 | HYDE: 0.13 | rfscore_v2: 0.88 | CNNaffinity: 0.78 | vina_intra_hydrophobic: 0.85 | SCORCH: 0.26 | RTMScore: 0.76 | CHEMPLP: 0.81 | CNNscore: 0.19
8,diffdock vs gnina,0.463,10,smina_affinity: -0.24 | vina_hydrophobic: 0.32 | HYDE: 0.13 | rfscore_v2: 0.87 | CNNaffinity: 0.78 | vina_intra_hydrophobic: 0.84 | SCORCH: 0.23 | RTMScore: 0.72 | CHEMPLP: 0.79 | CNNscore: 0.18
9,diffdock vs flexx,0.438,10,smina_affinity: 0.04 | vina_hydrophobic: 0.13 | HYDE: 0.10 | rfscore_v2: 0.78 | CNNaffinity: 0.62 | vina_intra_hydrophobic: 0.87 | SCORCH: 0.32 | RTMScore: 0.42 | CHEMPLP: 0.81 | CNNscore: 0.29


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV

# 1. Define final chosen ensemble columns (From Row 11: flexx + plants + smina)
final_features = [
    'smina_CNNscore', 'smina_CNNaffinity', 'plants_RTMScore', 
    'smina_RTMScore', 'plants_SCORCH', 'smina_SCORCH', 
    'flexx_vina_hydrophobic', 'smina_vina_hydrophobic', 
    'flexx_vina_intra_hydrophobic', 'plants_vina_intra_hydrophobic'
]

X_final = df_matrix[final_features]
y = df_matrix["true_value"]

# 2. Set up the hyperparameter grid
param_grid = {
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 300, 500, 700],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.5, 1.0] 
}

# 3. Define the base model and Cross-Validation
xgb_base = XGBRegressor(random_state=42, n_jobs=-1)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 4. Run the Search
print(f"Starting Hyperparameter Tuning on our 3-Tool Ensemble ({len(final_features)} features)...")
print("Running 100 different model configurations (this may take a minute)...")

tuner = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=100, 
    scoring='neg_mean_squared_error',
    cv=cv,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

tuner.fit(X_final, y)

# 5. Extract and format the results
best_rmse = np.sqrt(-tuner.best_score_)
best_params = tuner.best_params_

print("\n" + "="*40)
print("🏆 FINAL TUNING COMPLETE 🏆")
print("="*40)
print(f"Old Ensemble Baseline RMSE: ~1.3549")
print(f"New Optimized RMSE: {best_rmse:.4f}")
print("-" * 40)
print("Best Hyperparameters to report in your project:")
for param, value in best_params.items():
    print(f" - {param}: {value}")
print("="*40)

Starting Hyperparameter Tuning on our 3-Tool Ensemble (10 features)...
Running 100 different model configurations (this may take a minute)...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

🏆 FINAL TUNING COMPLETE 🏆
Old Ensemble Baseline RMSE: ~1.3549
New Optimized RMSE: 1.3235
----------------------------------------
Best Hyperparameters to report in your project:
 - subsample: 0.6
 - n_estimators: 300
 - max_depth: 6
 - learning_rate: 0.01
 - gamma: 0.5
 - colsample_bytree: 0.6


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import pearsonr, spearmanr

# 1. Define the exact columns for our 3 paradigms (Based on  previous ML Feature Selection)
paradigms = {
    "1. Unlimited Budget (Diffdock+Gnina)": {
        "cost": 520.44,
        "features": ['gnina_smina_affinity', 'diffdock_RTMScore', 'gnina_RTMScore', 'gnina_SCORCH', 'gnina_vina_hydrophobic', 'diffdock_vina_intra_hydrophobic'],
        "tuned": False
    },
    "2. Unlimited Budget (tuned Diffdock+Gnina)": {
        "cost": 520.44,
        "features": ['gnina_smina_affinity', 'diffdock_RTMScore', 'gnina_RTMScore', 'gnina_SCORCH', 'gnina_vina_hydrophobic', 'diffdock_vina_intra_hydrophobic'],
        "tuned": True
    },
    "3. Naive Budget (Smina Standalone)": {
        "cost": 105.63,
        "features": ['smina_RTMScore', 'smina_SCORCH', 'smina_vina_hydrophobic'],
        "tuned": False
    },
    "4. Engineered Masterpiece (Flexx+Plants+Smina)": {
        "cost": 123.54,
        "features": ['smina_CNNscore', 'smina_CNNaffinity', 'plants_RTMScore', 'smina_RTMScore', 'plants_SCORCH', 'smina_SCORCH', 'flexx_vina_hydrophobic', 'smina_vina_hydrophobic', 'flexx_vina_intra_hydrophobic', 'plants_vina_intra_hydrophobic'],
        "tuned": False
    },
    "5. Engineered Masterpiece (Tuned Flexx+Plants+Smina)": {
        "cost": 123.54,
        "features": ['smina_CNNscore', 'smina_CNNaffinity', 'plants_RTMScore', 'smina_RTMScore', 'plants_SCORCH', 'smina_SCORCH', 'flexx_vina_hydrophobic', 'smina_vina_hydrophobic', 'flexx_vina_intra_hydrophobic', 'plants_vina_intra_hydrophobic'],
        "tuned": True
    }
}

y = df_matrix["true_value"]
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Untuned Base Model
xgb_untuned = XGBRegressor(max_depth=4, learning_rate=0.05, n_estimators=300, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)

# Custom Tuned Model!
xgb_tuned = XGBRegressor(max_depth=6, learning_rate=0.01, n_estimators=300, subsample=0.6, colsample_bytree=0.6, gamma=0.5, random_state=42, n_jobs=-1)

results = []

for name, details in paradigms.items():
    X_subset = df_matrix[details["features"]]
    model = xgb_tuned if details["tuned"] else xgb_untuned
    
    fold_rmse, fold_mae, fold_r2, fold_pearson, fold_spearman = [], [], [], [], []
    
    for train_idx, test_idx in cv.split(X_subset):
        X_train, X_test = X_subset.iloc[train_idx], X_subset.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        fold_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
        fold_mae.append(mean_absolute_error(y_test, y_pred))
        fold_r2.append(r2_score(y_test, y_pred))
        fold_pearson.append(pearsonr(y_test, y_pred)[0])
        fold_spearman.append(spearmanr(y_test, y_pred)[0])
        
    results.append({
        "Model Strategy": name,
        "Cost (Seconds)": details["cost"],
        "RMSE (Accuracy)": np.mean(fold_rmse),
        "R2 (Variance Explained)": np.mean(fold_r2),
        "Spearman (Ranking)": np.mean(fold_spearman),
        "MAE (Absolute Error)": np.mean(fold_mae)
    })

final_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
display(final_df)

,Model Strategy,Cost (Seconds),RMSE (Accuracy),R2 (Variance Explained),Spearman (Ranking),MAE (Absolute Error)
0,1. Unlimited Budget (Diffdock+Gnina),520.44,1.254446,0.235709,0.462310,1.003875
1,2. Unlimited Budget (tuned Diffdock+Gnina),520.44,1.217758,0.288468,0.493883,0.988888
2,3. Naive Budget (Smina Standalone),105.63,1.360384,0.111805,0.430937,1.098814
3,4. Engineered Masterpiece (Flexx+Plants+Smina),123.54,1.354926,0.110811,0.407825,1.078636
4,5. Engineered Masterpiece (Tuned Flexx+Plants+Smina),123.54,1.313454,0.172490,0.414990,1.069787


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer 

# 1. engineered masterpiece features
final_features = [
    'smina_CNNscore', 'smina_CNNaffinity', 'plants_RTMScore', 
    'smina_RTMScore', 'plants_SCORCH', 'smina_SCORCH', 
    'flexx_vina_hydrophobic', 'smina_vina_hydrophobic', 
    'flexx_vina_intra_hydrophobic', 'plants_vina_intra_hydrophobic'
]

X_final = df_matrix[final_features]
y = df_matrix["true_value"]

# 2. Define Best Tuned XGBoost Model
xgb_tuned = XGBRegressor(
    max_depth=6, learning_rate=0.01, n_estimators=300, 
    subsample=0.6, colsample_bytree=0.6, gamma=0.5, 
    random_state=42, n_jobs=-1
)

# 3. Create the FIXED PCA Pipeline
pca_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')), # Fills NaNs with the column average
    ('scaler', StandardScaler()),                # Scales data for PCA
    ('pca', PCA(n_components=0.95, random_state=42)), # Compresses features
    ('xgb', xgb_tuned)                           # Makes the prediction
])

# 4. Cross-Validation Setup
cv = KFold(n_splits=5, shuffle=True, random_state=42)
fold_rmse_no_pca = []
fold_rmse_with_pca = []
components_used = []

print("Running Head-to-Head: Raw Features vs. PCA Compressed Features...")

for train_idx, test_idx in cv.split(X_final):
    X_train, X_test = X_final.iloc[train_idx], X_final.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # --- Test 1: WITHOUT PCA ( baseline) ---
    xgb_tuned.fit(X_train, y_train)
    y_pred_no_pca = xgb_tuned.predict(X_test)
    fold_rmse_no_pca.append(np.sqrt(mean_squared_error(y_test, y_pred_no_pca)))
    
    # --- Test 2: WITH PCA ---
    pca_pipeline.fit(X_train, y_train)
    y_pred_pca = pca_pipeline.predict(X_test)
    fold_rmse_with_pca.append(np.sqrt(mean_squared_error(y_test, y_pred_pca)))
    
    # Check how many columns PCA squashed the data into
    components_used.append(pca_pipeline.named_steps['pca'].n_components_)

# 5. Results
mean_rmse_no_pca = np.mean(fold_rmse_no_pca)
mean_rmse_pca = np.mean(fold_rmse_with_pca)
avg_components = np.mean(components_used)

print("\n" + "="*50)
print("🧬 PCA DIMENSIONALITY REDUCTION RESULTS 🧬")
print("="*50)
print(f"Original Columns Used: 10")
print(f"RMSE (Without PCA):    {mean_rmse_no_pca:.4f}")
print("-" * 50)
print(f"PCA Components Used:   ~{avg_components:.1f} (Retaining 95% of variance)")
print(f"RMSE (With PCA):       {mean_rmse_pca:.4f}")
print("="*50)

Running Head-to-Head: Raw Features vs. PCA Compressed Features...

🧬 PCA DIMENSIONALITY REDUCTION RESULTS 🧬
Original Columns Used: 10
RMSE (Without PCA):    1.3135
--------------------------------------------------
PCA Components Used:   ~8.0 (Retaining 95% of variance)
RMSE (With PCA):       1.3700


In [ ]:
import pandas as pd
import numpy as np

print("Loading and Aggregating ECF-T Data...")
df_orig_ecft = pd.read_csv("experiments/ecft_data/all_rescoring_results_ecft.csv")

# adjust the aggregation rules for the new scoring functions.
# Assuming LinF9 is empirical (lower is better -> min) and rfscores are random forest (higher is better -> max)
agg_rules_ecft = {
    'CNNscore': 'max', 'CNNaffinity': 'max', 'RTMScore': 'max', 
    'SCORCH': 'max', 'HYDE': 'max', 'rfscore_v1': 'max', 'rfscore_v3': 'max',
    'CHEMPLP': 'min', 'vina_hydrophobic': 'min', 'vina_intra_hydrophobic': 'min', 'LinF9': 'min',
    'true_value': 'first', 'activity_class': 'first'
}

# Catch-all for any other numeric columns
for col in df_orig_ecft.select_dtypes(include=np.number).columns:
    if col not in agg_rules_ecft and col not in ["pose", "id", "ID", "true_value", "activity_class"]:
        agg_rules_ecft[col] = "mean"

# Group by molecule ID and Docking Tool
df_agg_ecft = df_orig_ecft.groupby(["id", "docking_tool"]).agg(agg_rules_ecft).reset_index()

meta_cols = ["true_value", "activity_class"]
score_cols = [c for c in df_agg_ecft.columns if c not in meta_cols and c not in ["id", "docking_tool"]]

# Pivot the table to be wide
wide_ecft = df_agg_ecft.set_index(["id", "docking_tool"])[score_cols].unstack("docking_tool")
wide_ecft.columns = [f"{tool}_{score}" for score, tool in wide_ecft.columns]

# Re-attach the true binding affinities
meta_ecft = df_agg_ecft.groupby("id")[meta_cols].first()
df_matrix_ecft = wide_ecft.join(meta_ecft).dropna(subset=["true_value"])

print("ECF-T Data successfully transformed into df_matrix_ecft!")

Loading and Aggregating ECF-T Data...
ECF-T Data successfully transformed into df_matrix_ecft!


In [11]:
df_matrix_ecft.columns

Index(['GNINA_CNNscore', 'PLANTS_CNNscore', 'SMINA_CNNscore', 'flexx_CNNscore',
       'localdiffdock_CNNscore', 'GNINA_CNNaffinity', 'PLANTS_CNNaffinity',
       'SMINA_CNNaffinity', 'flexx_CNNaffinity', 'localdiffdock_CNNaffinity',
       'GNINA_RTMScore', 'PLANTS_RTMScore', 'SMINA_RTMScore', 'flexx_RTMScore',
       'localdiffdock_RTMScore', 'GNINA_SCORCH', 'PLANTS_SCORCH',
       'SMINA_SCORCH', 'flexx_SCORCH', 'localdiffdock_SCORCH', 'GNINA_HYDE',
       'PLANTS_HYDE', 'SMINA_HYDE', 'flexx_HYDE', 'localdiffdock_HYDE',
       'GNINA_rfscore_v1', 'PLANTS_rfscore_v1', 'SMINA_rfscore_v1',
       'flexx_rfscore_v1', 'localdiffdock_rfscore_v1', 'GNINA_rfscore_v3',
       'PLANTS_rfscore_v3', 'SMINA_rfscore_v3', 'flexx_rfscore_v3',
       'localdiffdock_rfscore_v3', 'GNINA_CHEMPLP', 'PLANTS_CHEMPLP',
       'SMINA_CHEMPLP', 'flexx_CHEMPLP', 'localdiffdock_CHEMPLP',
       'GNINA_vina_hydrophobic', 'PLANTS_vina_hydrophobic',
       'SMINA_vina_hydrophobic', 'flexx_vina_hydrophobic',
     

In [12]:
import pandas as pd
import numpy as np
import itertools
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr
from sklearn.feature_selection import SelectFromModel

# 1. Costs Dictionary
costs = {
    "localdiffdock": 407.5, "diffdock": 407.5, "flexx": 3.33, "smina": 99.9,
    "gnina": 105.8, "plants": 6.85, "cnnscore": 0.31, "cnnaffinity": 0.31,
    "smina_affinity": 0.31, "ad4": 0.28, "linf9": 0.24, "rtmscore": 0.41,
    "vinardo": 0.29, "scorch": 4.63, "hyde": 2.0, "chemplp": 0.121,
    "rfscore_v1": 0.682, "rfscore_v2": 0.687, "rfscore_v3": 0.69,
    "vina_hydrophobic": 0.69, "vina_intra_hydrophobic": 0.69,
}

# 2. FIX: Force all feature columns to lowercase to ensure perfect string matching
df_matrix_ecft.columns = [col.lower() if col not in ['true_value', 'activity_class'] else col for col in df_matrix_ecft.columns]

X_full = df_matrix_ecft.drop(columns=["true_value", "activity_class"])
y = df_matrix_ecft["true_value"]

# FIX: Use 'localdiffdock' to match the dataset
base_tools = ['localdiffdock', 'flexx', 'gnina', 'plants', 'smina']

# 3. Model & CV Setup
xgb_model = XGBRegressor(
    max_depth=4, learning_rate=0.05, n_estimators=300,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

def calculate_smart_cost(selected_cols):
    total_cost = 0
    used_tools = set()
    for col in selected_cols:
        parts = col.split('_', 1) # splits 'gnina_cnnscore' into 'gnina' and 'cnnscore'
        if len(parts) == 2:
            tool, score = parts[0], parts[1]
            used_tools.add(tool)
            total_cost += costs.get(score, 0)
    for tool in used_tools:
        total_cost += costs.get(tool, 0)
    return total_cost

# 4. Generate Combinations and Evaluate
results = []
print("Starting Clean Baseline Pipeline on ECF-T Dataset...")

for r in range(1, len(base_tools) + 1):
    for combo in itertools.combinations(base_tools, r):
        # Keep display clean by renaming localdiffdock back to diffdock for the printout
        combo_name = " + ".join(combo).replace('localdiffdock', 'diffdock')
        
        subset_cols = [col for col in X_full.columns if any(col.startswith(tool + '_') for tool in combo)]
        if not subset_cols: continue
        X_combo = X_full[subset_cols].copy()
                    
        # ML Feature Selection
        selector_model = XGBRegressor(max_depth=3, n_estimators=100, random_state=42, n_jobs=-1)
        selector_model.fit(X_combo, y)
        
        selector = SelectFromModel(selector_model, prefit=True, threshold='mean')
        selected_mask = selector.get_support()
        selected_features = X_combo.columns[selected_mask].tolist()
        
        if len(selected_features) == 0: selected_features = X_combo.columns.tolist()
            
        X_selected = X_combo[selected_features]
        combo_cost = calculate_smart_cost(selected_features)
        
        fold_rmse, fold_r2, fold_spearman = [], [], []
        
        for train_idx, test_idx in cv.split(X_selected):
            X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            xgb_model.fit(X_train, y_train)
            y_pred = xgb_model.predict(X_test)
            
            fold_rmse.append(np.sqrt(mean_squared_error(y_test, y_pred)))
            fold_r2.append(r2_score(y_test, y_pred))
            fold_spearman.append(spearmanr(y_test, y_pred)[0])
            
        results.append({
            "Tool Combination": combo_name,
            "Cost (s)": round(combo_cost, 2),
            "Features Kept": len(selected_features),
            "RMSE": np.mean(fold_rmse),
            "R2": np.mean(fold_r2),
            "Spearman": np.mean(fold_spearman),
            "Selected Columns": selected_features 
        })

# 5. Output
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="RMSE", ascending=True).reset_index(drop=True)

print("\n--- ECF-T Validation Complete ---")
pd.set_option('display.max_colwidth', None)
display(results_df)

Starting Clean Baseline Pipeline on ECF-T Dataset...

--- ECF-T Validation Complete ---


,Tool Combination,Cost (s),Features Kept,RMSE,R2,Spearman,Selected Columns
0,diffdock + flexx,414.20,6,0.259589,0.501913,0.698049,"[flexx_cnnaffinity, localdiffdock_cnnaffinity, flexx_rfscore_v1, flexx_rfscore_v3, flexx_vina_hydrophobic, flexx_vina_intra_hydrophobic]"
1,diffdock + flexx + gnina,531.94,10,0.259902,0.503147,0.693629,"[flexx_cnnaffinity, localdiffdock_cnnaffinity, gnina_scorch, localdiffdock_scorch, gnina_hyde, gnina_rfscore_v1, flexx_rfscore_v1, localdiffdock_rfscore_v1, flexx_vina_hydrophobic, flexx_vina_intra_hydrophobic]"
2,diffdock + flexx + smina,520.79,10,0.260777,0.498485,0.680363,"[flexx_cnnaffinity, localdiffdock_cnnaffinity, localdiffdock_scorch, flexx_rfscore_v1, localdiffdock_rfscore_v1, flexx_rfscore_v3, localdiffdock_rfscore_v3, smina_vina_hydrophobic, flexx_vina_hydrophobic, flexx_vina_intra_hydrophobic]"
3,flexx + gnina,112.87,6,0.266674,0.473259,0.669250,"[flexx_cnnaffinity, gnina_rfscore_v1, flexx_rfscore_v1, flexx_rfscore_v3, flexx_vina_hydrophobic, flexx_vina_intra_hydrophobic]"
4,diffdock + flexx + plants + smina,525.32,11,0.267093,0.473524,0.693637,"[plants_cnnaffinity, smina_cnnaffinity, flexx_cnnaffinity, plants_hyde, flexx_rfscore_v1, localdiffdock_rfscore_v1, flexx_rfscore_v3, localdiffdock_rfscore_v3, flexx_vina_hydrophobic, localdiffdock_vina_hydrophobic, flexx_vina_intra_hydrophobic]"
5,flexx,6.39,5,0.267997,0.469299,0.666582,"[flexx_cnnaffinity, flexx_rfscore_v1, flexx_rfscore_v3, flexx_vina_hydrophobic, flexx_vina_intra_hydrophobic]"
6,diffdock + flexx + plants,429.74,11,0.268522,0.470332,0.688111,"[plants_cnnaffinity, flexx_cnnaffinity, plants_scorch, plants_hyde, flexx_rfscore_v1, localdiffdock_rfscore_v1, flexx_rfscore_v3, localdiffdock_rfscore_v3, flexx_vina_hydrophobic, localdiffdock_vina_hydrophobic, flexx_vina_intra_hydrophobic]"
7,flexx + plants,20.99,10,0.268791,0.471806,0.677966,"[plants_cnnaffinity, flexx_cnnaffinity, plants_scorch, plants_hyde, flexx_rfscore_v1, plants_rfscore_v3, flexx_rfscore_v3, flexx_chemplp, flexx_vina_hydrophobic, flexx_vina_intra_hydrophobic]"
8,flexx + plants + smina,126.08,11,0.268960,0.468843,0.665819,"[plants_cnnaffinity, flexx_cnnaffinity, plants_scorch, smina_scorch, plants_hyde, smina_rfscore_v1, flexx_rfscore_v1, plants_rfscore_v3, flexx_rfscore_v3, flexx_vina_hydrophobic, flexx_vina_intra_hydrophobic]"
9,diffdock + flexx + gnina + plants + smina,637.85,13,0.272627,0.455124,0.662772,"[plants_cnnaffinity, flexx_cnnaffinity, smina_rtmscore, smina_scorch, gnina_hyde, plants_hyde, gnina_rfscore_v1, localdiffdock_rfscore_v1, gnina_rfscore_v3, flexx_rfscore_v3, flexx_vina_hydrophobic, flexx_vina_intra_hydrophobic, localdiffdock_vina_intra_hydrophobic]"


In [17]:
import pandas as pd
import numpy as np
import itertools
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr
from sklearn.feature_selection import SelectFromModel

# ==========================================
# 1. LOAD AND PREPROCESS BOTH DATASETS
# ==========================================

# --- EGFR DATA (Train) ---
print("Loading EGFR (Train) Data...")
df_egfr_raw = pd.read_csv("all_rescoring_results_merged.csv")
agg_rules_egfr = {'CNNscore': 'max', 'CNNaffinity': 'max', 'RTMScore': 'max', 'SCORCH': 'max', 
                  'HYDE': 'max', 'rfscore_v2': 'max', 'CHEMPLP': 'min', 'vina_hydrophobic': 'min', 
                  'vina_intra_hydrophobic': 'min', 'smina_affinity': 'min', 'true_value': 'first'}
df_egfr = df_egfr_raw.groupby(["id", "docking_tool"]).agg({k:v for k,v in agg_rules_egfr.items() if k in df_egfr_raw.columns}).reset_index()
df_matrix_egfr = df_egfr.set_index(["id", "docking_tool"])[[c for c in df_egfr.columns if c not in ['id', 'docking_tool', 'true_value']]].unstack("docking_tool")
df_matrix_egfr.columns = [f"{tool.lower()}_{score}" for score, tool in df_matrix_egfr.columns]
df_matrix_egfr = df_matrix_egfr.join(df_egfr.groupby("id")[["true_value"]].first()).dropna()
print(df_egfr_raw.shape)

print(df_matrix_egfr.shape)

# --- ECF-T DATA (Test) ---
print("Loading ECF-T (Test) Data...")
df_ecft_raw = pd.read_csv("experiments/ecft_data/all_rescoring_results_ecft.csv")
# Replace 'localdiffdock' with 'diffdock' to match EGFR
print(df_ecft_raw.shape)

df_ecft_raw['docking_tool'] = df_ecft_raw['docking_tool'].str.replace('localdiffdock', 'diffdock', case=False)

agg_rules_ecft = {'CNNscore': 'max', 'CNNaffinity': 'max', 'RTMScore': 'max', 'SCORCH': 'max', 
                  'HYDE': 'max', 'rfscore_v1': 'max', 'rfscore_v3': 'max', 'CHEMPLP': 'min', 
                  'vina_hydrophobic': 'min', 'vina_intra_hydrophobic': 'min', 'LinF9': 'min', 'true_value': 'first'}
df_ecft = df_ecft_raw.groupby(["id", "docking_tool"]).agg({k:v for k,v in agg_rules_ecft.items() if k in df_ecft_raw.columns}).reset_index()
df_matrix_ecft = df_ecft.set_index(["id", "docking_tool"])[[c for c in df_ecft.columns if c not in ['id', 'docking_tool', 'true_value']]].unstack("docking_tool")
df_matrix_ecft.columns = [f"{tool.lower()}_{score}" for score, tool in df_matrix_ecft.columns]
df_matrix_ecft = df_matrix_ecft.join(df_ecft.groupby("id")[["true_value"]].first()).dropna()
print(df_matrix_ecft.shape)
# ==========================================
# 2. FIND THE INTERSECTION (Shared Columns)
# ==========================================
shared_features = list(set(df_matrix_egfr.columns) & set(df_matrix_ecft.columns))
shared_features.remove('true_value') # Keep targets separate

print(f"Found {len(shared_features)} overlapping features between EGFR and ECF-T.")

X_train_full = df_matrix_egfr[shared_features]
y_train = df_matrix_egfr['true_value']

X_test_full = df_matrix_ecft[shared_features]
y_test = df_matrix_ecft['true_value']

base_tools = ['diffdock', 'flexx', 'gnina', 'plants', 'smina']

# ==========================================
# 3. CROSS-TARGET EVALUATION (Train EGFR -> Predict ECF-T)
# ==========================================
results = []
print("Starting Zero-Shot Domain Generalization...")

for r in range(1, len(base_tools) + 1):
    for combo in itertools.combinations(base_tools, r):
        combo_name = " + ".join(combo)
        
        # Get only the shared columns that belong to the current tool combination
        subset_cols = [col for col in shared_features if any(col.startswith(tool + '_') for tool in combo)]
        if not subset_cols: continue
        
        X_train_combo = X_train_full[subset_cols].copy()
        X_test_combo = X_test_full[subset_cols].copy()
                    
        # Feature Selection (Learned entirely on EGFR)
        selector_model = XGBRegressor(max_depth=3, n_estimators=100, random_state=42, n_jobs=-1)
        selector_model.fit(X_train_combo, y_train)
        
        selector = SelectFromModel(selector_model, prefit=True, threshold='mean')
        selected_mask = selector.get_support()
        selected_features = X_train_combo.columns[selected_mask].tolist()
        
        if len(selected_features) == 0: selected_features = X_train_combo.columns.tolist()
            
        # Apply the exact same features to both Train and Test
        X_train_final = X_train_combo[selected_features]
        X_test_final = X_test_combo[selected_features]
        
        # Train on EGFR
        model = XGBRegressor(max_depth=4, learning_rate=0.05, n_estimators=300, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
        model.fit(X_train_final, y_train)
        
        # Predict on ECF-T (Zero-Shot)
        y_pred = model.predict(X_test_final)
        
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        spearman = spearmanr(y_test, y_pred)[0]
            
        results.append({
            "Tool Combination": combo_name,
            "Shared Features Kept": len(selected_features),
            "Zero-Shot RMSE": rmse,
            "Zero-Shot R2": r2,
            "Zero-Shot Spearman": spearman,
        })

# 4. Output
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Zero-Shot R2", ascending=False).reset_index(drop=True)

print("\n--- Cross-Target Generalization Complete ---")
display(results_df)

Loading EGFR (Train) Data...
(9756, 15)
(179, 51)
Loading ECF-T (Test) Data...
(10551, 16)
(209, 56)
Found 40 overlapping features between EGFR and ECF-T.
Starting Zero-Shot Domain Generalization...

--- Cross-Target Generalization Complete ---


,Tool Combination,Shared Features Kept,Zero-Shot RMSE,Zero-Shot R2,Zero-Shot Spearman
0,diffdock,1,3.442500,-84.060323,NaN
1,diffdock + gnina + plants + smina,8,3.724083,-98.544588,0.063641
2,diffdock + gnina + smina,6,4.006502,-114.215207,0.022783
3,diffdock + flexx,2,4.092704,-119.226382,0.107981
4,diffdock + gnina + plants,8,4.384385,-136.973761,-0.156557
5,diffdock + flexx + gnina,7,4.416939,-139.030249,-0.260819
6,diffdock + flexx + gnina + plants + smina,13,4.429758,-139.844222,0.187619
7,diffdock + flexx + gnina + plants,5,4.458008,-141.646373,-0.184620
8,diffdock + plants + smina,9,4.500911,-144.405165,0.059694
9,diffdock + flexx + plants + smina,7,4.575752,-149.280987,0.054963


In [ ]:
import pandas as pd
import numpy as np
import itertools
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.feature_selection import SelectFromModel

# ==========================================
# 1. LOAD AND PREPROCESS BOTH DATASETS
# ==========================================

# --- EGFR DATA (Train) ---
print("Loading EGFR (Train) Data...")
df_egfr_raw = pd.read_csv("all_rescoring_results_merged.csv")
agg_rules_egfr = {'CNNscore': 'max', 'CNNaffinity': 'max', 'RTMScore': 'max', 'SCORCH': 'max', 
                  'HYDE': 'max', 'rfscore_v2': 'max', 'CHEMPLP': 'min', 'vina_hydrophobic': 'min', 
                  'vina_intra_hydrophobic': 'min', 'smina_affinity': 'min', 'activity_class': 'first'}

df_egfr = df_egfr_raw.groupby(["id", "docking_tool"]).agg({k:v for k,v in agg_rules_egfr.items() if k in df_egfr_raw.columns}).reset_index()
df_matrix_egfr = df_egfr.set_index(["id", "docking_tool"])[[c for c in df_egfr.columns if c not in ['id', 'docking_tool', 'activity_class']]].unstack("docking_tool")
df_matrix_egfr.columns = [f"{tool.lower()}_{score}" for score, tool in df_matrix_egfr.columns]
df_matrix_egfr = df_matrix_egfr.join(df_egfr.groupby("id")[["activity_class"]].first()).dropna()

# --- ECF-T DATA (Test) ---
print("Loading ECF-T (Test) Data...")
df_ecft_raw = pd.read_csv("experiments/ecft_data/all_rescoring_results_ecft.csv")
df_ecft_raw['docking_tool'] = df_ecft_raw['docking_tool'].str.replace('localdiffdock', 'diffdock', case=False)

agg_rules_ecft = {'CNNscore': 'max', 'CNNaffinity': 'max', 'RTMScore': 'max', 'SCORCH': 'max', 
                  'HYDE': 'max', 'rfscore_v1': 'max', 'rfscore_v3': 'max', 'CHEMPLP': 'min', 
                  'vina_hydrophobic': 'min', 'vina_intra_hydrophobic': 'min', 'LinF9': 'min', 'activity_class': 'first'}

df_ecft = df_ecft_raw.groupby(["id", "docking_tool"]).agg({k:v for k,v in agg_rules_ecft.items() if k in df_ecft_raw.columns}).reset_index()
df_matrix_ecft = df_ecft.set_index(["id", "docking_tool"])[[c for c in df_ecft.columns if c not in ['id', 'docking_tool', 'activity_class']]].unstack("docking_tool")
df_matrix_ecft.columns = [f"{tool.lower()}_{score}" for score, tool in df_matrix_ecft.columns]
df_matrix_ecft = df_matrix_ecft.join(df_ecft.groupby("id")[["activity_class"]].first()).dropna()

# Force Activity Class to integer type (0 or 1)
df_matrix_egfr['activity_class'] = df_matrix_egfr['activity_class'].astype(int)
df_matrix_ecft['activity_class'] = df_matrix_ecft['activity_class'].astype(int)

# ==========================================
# 2. FIND THE INTERSECTION (Shared Columns)
# ==========================================
shared_features = list(set(df_matrix_egfr.columns) & set(df_matrix_ecft.columns))
shared_features.remove('activity_class')

print(f"Found {len(shared_features)} overlapping features between EGFR and ECF-T.")

X_train_full = df_matrix_egfr[shared_features]
y_train = df_matrix_egfr['activity_class']

X_test_full = df_matrix_ecft[shared_features]
y_test = df_matrix_ecft['activity_class']
# --- Check Class Imbalances ---
print("\n" + "="*40)
print("📊 CLASS IMBALANCE REPORT")
print("="*40)

# Train Data (EGFR)
train_counts = y_train.value_counts()
train_neg = train_counts.get(0, 0)
train_pos = train_counts.get(1, 0)
print(f"EGFR (Train) -> Inactives (0): {train_neg} | Actives (1): {train_pos}")
print(f"EGFR Ratio: {train_neg / train_pos:.2f} to 1")

# Test Data (ECF-T)
test_counts = y_test.value_counts()
test_neg = test_counts.get(0, 0)
test_pos = test_counts.get(1, 0)
print(f"ECF-T (Test) -> Inactives (0): {test_neg} | Actives (1): {test_pos}")
print(f"ECF-T Ratio: {test_neg / test_pos:.2f} to 1")
print("="*40 + "\n")
base_tools = ['diffdock', 'flexx', 'gnina', 'plants', 'smina']

# ==========================================
# 3. CROSS-TARGET EVALUATION (CLASSIFICATION)
# ==========================================
results = []
print("Starting Zero-Shot Domain Generalization (Classification)...")

for r in range(1, len(base_tools) + 1):
    for combo in itertools.combinations(base_tools, r):
        combo_name = " + ".join(combo)
        
        subset_cols = [col for col in shared_features if any(col.startswith(tool + '_') for tool in combo)]
        if not subset_cols: continue
        
        X_train_combo = X_train_full[subset_cols].copy()
        X_test_combo = X_test_full[subset_cols].copy()
                    
        # Feature Selection (Using a Classifier now!)
        selector_model = XGBClassifier(max_depth=3, n_estimators=100, random_state=42, n_jobs=-1, eval_metric='logloss', scale_pos_weight = 10)
        selector_model.fit(X_train_combo, y_train)
        
        selector = SelectFromModel(selector_model, prefit=True, threshold='mean')
        selected_mask = selector.get_support()
        selected_features = X_train_combo.columns[selected_mask].tolist()
        
        if len(selected_features) == 0: selected_features = X_train_combo.columns.tolist()
            
        X_train_final = X_train_combo[selected_features]
        X_test_final = X_test_combo[selected_features]
        
        # Train Main Classifier on EGFR
        model = XGBClassifier(max_depth=4, learning_rate=0.05, n_estimators=300, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, eval_metric='logloss', scale_pos_weight = 10)
        model.fit(X_train_final, y_train)
        
        # Predict on ECF-T (Zero-Shot)
        y_pred = model.predict(X_test_final)                 # Exact 0 or 1 predictions
        y_pred_proba = model.predict_proba(X_test_final)[:, 1] # Probability for ROC-AUC
        
        # Calculate Metrics
        try:
            auc = roc_auc_score(y_test, y_pred_proba)
        except ValueError:
            auc = np.nan # In case a dataset only has 1 class somehow
            
        f1 = f1_score(y_test, y_pred, average='macro')
        acc = accuracy_score(y_test, y_pred)
            
        results.append({
            "Tool Combination": combo_name,
            "Shared Features Kept": len(selected_features),
            "ROC-AUC": auc,
            "F1-Score": f1,
            "Accuracy": acc
        })

# 4. Output
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)

print("\n--- Cross-Target Classification Complete ---")
display(results_df)

Loading EGFR (Train) Data...
Loading ECF-T (Test) Data...
Found 40 overlapping features between EGFR and ECF-T.
Starting Zero-Shot Domain Generalization (Classification)...

--- Cross-Target Classification Complete ---


,Tool Combination,Shared Features Kept,ROC-AUC,F1-Score,Accuracy
0,gnina + smina,5,0.668160,0.425824,0.741627
1,diffdock + flexx + plants,8,0.665472,0.425824,0.741627
2,plants + smina,7,0.590502,0.425824,0.741627
3,flexx + plants,6,0.547312,0.462584,0.717703
4,diffdock + flexx + gnina + plants + smina,13,0.544205,0.425824,0.741627
5,flexx,5,0.482437,0.505282,0.679426
6,diffdock + flexx,6,0.478375,0.425824,0.741627
7,plants,4,0.477001,0.421053,0.727273
8,diffdock + flexx + gnina + smina,10,0.475568,0.425824,0.741627
9,diffdock + gnina + plants + smina,10,0.469654,0.425824,0.741627
